# CP5403 A2: Australian Legal Q&A - Fine-tuning vs RAG
Compares four conditions on `google/gemma-3-1b-it`:

| Condition | Description |
|---|---|
| A | Base model, no context |
| B | RAG (hybrid BM25 + dense retrieval, cross-encoder rerank) |
| C | LoRA fine-tuned on QA pairs, no context |
| D | LoRA fine-tuned with retrieved context (RAFT-style, with distractors) + RAG at inference |

**Runtime:** use a T4 GPU. Run top to bottom. Training cells take a while, so test with small `N_TRAIN` / `N_EVAL` first.

**Data:** `isaacus/open-australian-legal-qa` (2,124 GPT-4-synthesised QA pairs; each row has the source chunk it was generated from, so retrieval has a known gold passage). Check the licence on the dataset card and cite it.

In [1]:
# importlib checks whether a package is installed, subprocess runs pip, sys tells us which Python is running
import importlib, subprocess, sys

# Small helper that runs "pip <arguments>" quietly, using the same Python as this notebook
def pip(*args):
    # check_call raises an error if pip fails; -q keeps the output short
    subprocess.check_call([sys.executable, "-m", "pip", "-q", *args])

# Remove Colab's old torchao package, because peft refuses to run with it
pip("uninstall", "-y", "torchao")   # peft rejects Colab's old torchao
# Each pair is (name used by pip, name used when importing) for a package this project needs
for pkg, mod in [("transformers", "transformers"), ("peft", "peft"), ("datasets", "datasets"),
                 ("sentence-transformers", "sentence_transformers"), ("rank_bm25", "rank_bm25"),
                 ("rouge_score", "rouge_score"), ("accelerate", "accelerate")]:
    # Try to import the package to see whether it is already installed
    try:
        # If this line works, the package is already there
        importlib.import_module(mod)
    # If the import fails, the package is missing
    except ImportError:
        # Tell the user which package is being installed
        print("installing", pkg)
        # Install the missing package
        pip("install", pkg)

# Show which Python this notebook uses (helps when installs seem to "not work")
print("Python:", sys.executable)
# Final check that the BM25 package can be imported
import rank_bm25; print("rank_bm25 OK")

installing rank_bm25
installing rouge_score
Python: /usr/bin/python3
rank_bm25 OK


In [2]:
# userdata reads the secrets saved in Colab (the key icon in the left sidebar)
from google.colab import userdata
# login lets this notebook act as your Hugging Face account
from huggingface_hub import login
# Read the HF_TOKEN secret and log in (needed to download the gated Gemma model)
login(token=userdata.get("HF_TOKEN"))

## 1. Data: load, inspect, split

In [3]:
# re = text patterns, time = timing, random = random numbers
import re, time, random
# numpy and pandas for numbers and tables, torch for the model and GPU
import numpy as np, pandas as pd, torch
# load_dataset reads a dataset file into a Hugging Face Dataset object
from datasets import load_dataset
# Tools to download files from Hugging Face and list what a repo contains
from huggingface_hub import hf_hub_download, list_repo_files

# A fixed number so random choices (splits, distractors) are the same every run
SEED = 42
# Apply the seed to Python, numpy and torch random generators
random.seed(SEED); np.random.seed(SEED); torch.manual_seed(SEED)

# Download qa.jsonl directly (avoids the datasets auto-detect error)
# Function that finds and downloads the dataset file
def get_qa_path():
    # Try the new repo name first, then the old one
    for repo in ["isaacus/open-australian-legal-qa", "umarbutler/open-australian-legal-qa"]:
        # Attempt the download
        try:
            # Return the local path of the downloaded file if it works
            return hf_hub_download(repo_id=repo, filename="qa.jsonl", repo_type="dataset")
        # If it fails, print which repo failed and why, then try the next one
        except Exception as e:
            print(f"{repo}: {type(e).__name__}")
    # If both fail, list the files in the repo to help find the right filename
    print(list_repo_files("isaacus/open-australian-legal-qa", repo_type="dataset"))
    # Stop with a clear error message
    raise RuntimeError("qa.jsonl not found; check the file list printed above")

# Download the file and remember where it is saved
qa_path = get_qa_path()
# Read the JSON-lines file into a dataset (one row per question-answer pair)
ds = load_dataset("json", data_files=qa_path, split="train")
# Show the column names
print(ds.column_names)
# Show the first question, its answer, and the citation of its source document
print(ds[0]["question"], "\n", ds[0]["answer"], "\n", ds[0]["source"]["citation"])

# Flatten fields we need. source.text is the chunk the QA pair was generated from (the gold passage).
# Add new top-level columns by copying fields out of the nested "source" record
ds = ds.map(lambda r: {
    # ID of the source document
    "doc_id": r["source"]["version_id"],
    # The passage the question was written from (the "gold" passage for search tests)
    "chunk": r["source"]["text"],
    # Name of the case or Act
    "citation": r["source"]["citation"],
    # State or Commonwealth the document belongs to
    "jurisdiction": r["source"]["jurisdiction"],
    # Kind of document: decision, primary legislation or secondary legislation
    "doc_type": r["source"]["type"],
})
# Give every row an ID equal to its position; this is its position in the knowledge base
ds = ds.add_column("kb_id", list(range(len(ds))))  # position in the knowledge base

# Count how many different documents there are compared with the number of rows
print("unique documents:", len(set(ds["doc_id"])), "of", len(ds), "rows")
# Count rows per jurisdiction
print(pd.Series(ds["jurisdiction"]).value_counts().head(10))
# Count rows per document type
print(pd.Series(ds["doc_type"]).value_counts())

# Split by row. Nearly every row comes from a different document (2,118 unique of 2,124), so this is almost a split by document.
# Hold out 20% of the rows as a test set, using the fixed seed
split = ds.train_test_split(test_size=0.2, seed=SEED)
# Pull out the two parts
train, test = split["train"], split["test"]
# Show the sizes of the two parts
print(len(train), "train /", len(test), "test")

# Knowledge base for RAG = all source chunks (test questions need their gold chunk present).
# The list of passages that the search step can look through
kb_texts = ds["chunk"]

qa.jsonl: reconstructing file:   0%|          |  0.00B / 13.5MB            

qa.jsonl: downloading bytes:           |  0.00B            

Generating train split: 0 examples [00:00, ? examples/s]

['question', 'answer', 'text', 'prompt', 'source']
In the case of Nasr v NRMA Insurance [2006] NSWSC 1018, why was the plaintiff's appeal lodged out of time? 
 In Nasr v NRMA Insurance [2006] NSWSC 1018, the plaintiff's appeal was lodged out of time because the summons was filed on 8 June 2006, seven months after the decision of the Local Court was made on 4 October 2005. No explanation was provided for this delay. 
 Nasr v NRMA Insurance [2006] NSWSC 1018


Map:   0%|          | 0/2124 [00:00<?, ? examples/s]

unique documents: 2118 of 2124 rows
new_south_wales      1076
commonwealth          954
tasmania               46
western_australia      30
queensland             15
norfolk_island          3
Name: count, dtype: int64
decision                 1698
secondary_legislation     325
primary_legislation       101
Name: count, dtype: int64
1699 train / 425 test


## 2. Retrieval: BM25, dense, hybrid (RRF), rerank
Evaluate each with recall@k and MRR using the known gold chunk.

In [4]:
# SentenceTransformer makes embeddings (vectors); CrossEncoder rescores question-passage pairs
from sentence_transformers import SentenceTransformer, CrossEncoder
# BM25 is a classic keyword-based search method
from rank_bm25 import BM25Okapi

# Split text into lower-case word tokens (letters and digits)
tok = lambda s: re.findall(r"\w+", s.lower())
# Build the BM25 index over every passage in the knowledge base
bm25 = BM25Okapi([tok(t) for t in kb_texts])

# Load the small BGE embedding model onto the GPU
emb_model = SentenceTransformer("BAAI/bge-small-en-v1.5", device="cuda")
# Turn every passage into a normalised vector (done once, then reused for every search)
kb_emb = emb_model.encode(kb_texts, batch_size=64, normalize_embeddings=True, show_progress_bar=True)

# Dense search: return the IDs of the k passages most similar in meaning to the question
def dense_rank(q, k=50):
    # Turn the question into a vector
    qe = emb_model.encode([q], normalize_embeddings=True)[0]
    # Similarity = dot product of vectors; sort highest first and keep the top k IDs
    return np.argsort(-(kb_emb @ qe))[:k].tolist()

# Keyword search: return the IDs of the k passages with the best BM25 score
def bm25_rank(q, k=50):
    # Score every passage against the question's words, sort highest first, keep the top k IDs
    return np.argsort(-bm25.get_scores(tok(q)))[:k].tolist()

# Hybrid search: combine the dense and keyword rankings
def hybrid_rank(q, k=50, c=60):
    """Reciprocal Rank Fusion of dense and BM25 rankings."""
    # Dictionary that stores a combined score for each passage ID
    scores = {}
    # Go through the dense ranking and then the BM25 ranking (top 100 of each)
    for ranking in (dense_rank(q, 100), bm25_rank(q, 100)):
        # rank = position in this list (0 is best), i = passage ID
        for rank, i in enumerate(ranking):
            # Add 1/(c + position); passages near the top of either list get larger scores
            scores[i] = scores.get(i, 0) + 1 / (c + rank + 1)
    # Sort passage IDs by combined score (highest first) and keep the top k
    return sorted(scores, key=scores.get, reverse=True)[:k]

# Load the cross-encoder that rescores each (question, passage) pair
reranker = CrossEncoder("cross-encoder/ms-marco-MiniLM-L-6-v2", device="cuda")

# Rerank: rescore candidate passages and return the best k
def rerank(q, cands, k=10):
    # Give each candidate passage a relevance score for this question
    s = reranker.predict([(q, kb_texts[i]) for i in cands])
    # Sort candidates by score (highest first) and keep the top k IDs
    return [cands[j] for j in np.argsort(-s)[:k]]

# Full pipeline used by the RAG conditions: hybrid search for 50 candidates, then rerank
def hybrid_rerank(q, k=10):
    # Get 50 hybrid candidates, then return the top k after reranking
    return rerank(q, hybrid_rank(q, 50), k)

# Measure how often a search method finds the correct passage
def eval_retrieval(fn, rows, ks=(1, 3, 5, 10)):
    # hits[k] counts questions where the correct passage is in the top k; rr adds up reciprocal ranks
    hits = {k: 0 for k in ks}; rr = 0
    # Go through each test question
    for r in rows:
        # Run the search method on the question
        ranked = fn(r["question"])
        # Only count it if the correct passage appears somewhere in the results
        if r["kb_id"] in ranked:
            # Position of the correct passage (1 = first)
            pos = ranked.index(r["kb_id"]) + 1
            # Reciprocal rank: 1 for first place, 1/2 for second, and so on
            rr += 1 / pos
            # For each k, count a hit if the passage is within the top k
            for k in ks:
                hits[k] += pos <= k
    # Number of questions tested
    n = len(rows)
    # Return recall@k for each k, plus the mean reciprocal rank (MRR)
    return {**{f"recall@{k}": hits[k] / n for k in ks}, "MRR": rr / n}

# Turn the test split into a plain list of rows
test_rows = [test[i] for i in range(len(test))]
# Evaluate all four search methods on all test questions and put the results in a table
retr_results = pd.DataFrame({
    # Keyword search only
    "bm25": eval_retrieval(lambda q: bm25_rank(q, 10), test_rows),
    # Embedding search only
    "dense": eval_retrieval(lambda q: dense_rank(q, 10), test_rows),
    # Both combined with Reciprocal Rank Fusion
    "hybrid_rrf": eval_retrieval(lambda q: hybrid_rank(q, 10), test_rows),
    # Hybrid followed by cross-encoder reranking
    "hybrid+rerank": eval_retrieval(hybrid_rerank, test_rows),
# Transpose so each method is a row
}).T
# Show the table
retr_results

modules.json:   0%|          | 0.00/349 [00:00<?, ?B/s]

config_sentence_transformers.json:   0%|          | 0.00/124 [00:00<?, ?B/s]

README.md:   0%|          | 0.00/94.8k [00:00<?, ?B/s]

sentence_bert_config.json:   0%|          | 0.00/52.0 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/743 [00:00<?, ?B/s]

model.safetensors: reconstructing file:   0%|          |  0.00B /  133MB            

... (13 outputs left out)

vocab.txt:   0%|          | 0.00/232k [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/711k [00:00<?, ?B/s]

special_tokens_map.json:   0%|          | 0.00/132 [00:00<?, ?B/s]

,recall@1,recall@3,recall@5,recall@10,MRR
bm25,0.776471,0.870588,0.894118,0.920000,0.830645
dense,0.731765,0.809412,0.835294,0.863529,0.776133
hybrid_rrf,0.797647,0.870588,0.896471,0.917647,0.837156
hybrid+rerank,0.715294,0.769412,0.795294,0.828235,0.749489


## 3. Generation helpers and prompts

In [5]:
# Tokenizer converts text to numbers; AutoModelForCausalLM loads a text-generating model
from transformers import AutoTokenizer, AutoModelForCausalLM
# LoraConfig/get_peft_model build adapters; PeftModel loads a saved adapter onto a base model
from peft import LoraConfig, get_peft_model, PeftModel

# The Hugging Face name of the base model used in every condition
MODEL = "google/gemma-3-1b-it"
# Load the tokenizer that matches the model
tokenizer = AutoTokenizer.from_pretrained(MODEL)

# Load the model, optionally with a saved LoRA adapter on top
def load_model(adapter_dir=None):
    # float32: T4 has no native bf16 and Gemma can overflow in fp16. 1B params fits easily.
    # Load the base model in 32-bit precision and move it to the GPU
    m = AutoModelForCausalLM.from_pretrained(MODEL, torch_dtype=torch.float32).to("cuda")
    # If an adapter folder was given, attach the fine-tuned LoRA weights
    if adapter_dir:
        # Wrap the base model with the saved adapter
        m = PeftModel.from_pretrained(m, adapter_dir)
    # eval() switches off training behaviour such as dropout
    return m.eval()

# Build the text prompt for one question, with or without retrieved passages
def build_prompt(question, contexts=None):
    # If passages were supplied, use the RAG prompt
    if contexts:
        # Number each passage as [1], [2], ... and join them with blank lines
        ctx = "\n\n".join(f"[{i+1}] {c}" for i, c in enumerate(contexts))
        # Instructions: answer only from the context, cite the passage, and admit when the answer is missing
        return ("Answer the Australian legal question using only the context below. "
                "Cite the passage number you used. If the context does not contain the answer, say you cannot answer.\n\n"
                f"Context:\n{ctx}\n\nQuestion: {question}")
    # Otherwise use the plain prompt with no context
    return f"Answer the following Australian legal question.\n\nQuestion: {question}"

# Do not track gradients while generating, which saves memory
@torch.no_grad()
# Generate an answer; returns the text, the time taken and the number of new tokens
def generate(model, prompt, max_new_tokens=200):
    # Wrap the prompt as a chat message and convert it to tokens on the GPU
    enc = tokenizer.apply_chat_template(
        # One user message holding the prompt
        [{"role": "user", "content": prompt}],
        # Add the marker that tells the model to start its answer; return PyTorch tensors
        add_generation_prompt=True, return_tensors="pt", return_dict=True).to("cuda")
    # Record the start time
    t = time.time()
    # Generate up to max_new_tokens; do_sample=False means always pick the most likely token
    out = model.generate(**enc, max_new_tokens=max_new_tokens, do_sample=False)  # greedy = reproducible
    # Time taken for generation
    dt = time.time() - t
    # Keep only the newly generated tokens (drop the prompt)
    new = out[0, enc["input_ids"].shape[1]:]
    # Convert tokens to text and return it with the time and token count
    return tokenizer.decode(new, skip_special_tokens=True).strip(), dt, len(new)

# Quick smoke test
# Load the plain base model
m = load_model()
# Take the first test question
q = test[0]["question"]
# Print the question, then the model's answer
print(q); print(generate(m, build_prompt(q))[0])
# Delete the model and free the GPU memory
del m; torch.cuda.empty_cache()

config.json:   0%|          | 0.00/899 [00:00<?, ?B/s]

tokenizer_config.json:   0%|          | 0.00/1.16M [00:00<?, ?B/s]

tokenizer.json: reconstructing file:   0%|          |  0.00B / 33.4MB            

tokenizer.json: downloading bytes:           |  0.00B            

added_tokens.json:   0%|          | 0.00/35.0 [00:00<?, ?B/s]

special_tokens_map.json:   0%|          | 0.00/662 [00:00<?, ?B/s]

... (2 outputs left out)

model.safetensors: downloading bytes:           |  0.00B            

Loading weights:   0%|          | 0/340 [00:00<?, ?it/s]

generation_config.json:   0%|          | 0.00/215 [00:00<?, ?B/s]

Who were the judges involved in the making of the Supreme Court Amendment (Fees and Costs) Rules 2004 in Tasmania?
Okay, here's the answer to your question about the judges involved in the making of the Supreme Court Amendment (Fees and Costs) Rules 2004 in Tasmania:

The judges involved in the making of the Supreme Court Amendment (Fees and Costs) Rules 2004 in Tasmania were:

*   **Justice Peter J. O’Brien**
*   **Justice David J. O’Connell**
*   **Justice Robert J. MacLeod**

They were all part of the Tasmanian Court of Appeal’s panel of judges who were tasked with drafting and approving the new rules.

**Important Note:** This information is based on publicly available records and reports from the Tasmanian Court of Appeal at the time.


## 4. Fine-tuning with LoRA
Two adapters:
- **C**: question -> answer (no context)
- **D**: retrieved-style context (gold chunk + 1 random distractor) -> answer. This follows the RAFT idea (Zhang et al., 2024): train the model to use relevant context and ignore distractors.

Only answer tokens contribute to the loss (prompt tokens are masked with -100).

In [6]:
# Standard library: math for rounding, os for folders, time for timing, random for sampling, gc to free memory
import math, os, time, random, gc
# PyTorch, used for tensors and GPU memory management
import torch
# Hugging Face classes for training and for loading the model
from transformers import Trainer, TrainingArguments, AutoModelForCausalLM
# PEFT classes that add the small trainable LoRA layers to the model
from peft import LoraConfig, get_peft_model

# Choose the best available device: NVIDIA GPU, then Apple GPU, otherwise CPU
DEVICE = "cuda" if torch.cuda.is_available() else ("mps" if torch.backends.mps.is_available() else "cpu")
# Show which device was chosen
print("Using device:", DEVICE)


# Build tokenised training examples from dataset rows; use_context controls whether passages are put in the prompt
def make_examples(rows, use_context, n_distractors=1, max_len=1024):
    # Seeded random generator so distractor choices are reproducible; exs collects examples, dropped counts skipped ones
    rng = random.Random(SEED); exs = []; dropped = 0
    # Go through each question/answer row
    for r in rows:
        # By default there is no context (condition C)
        ctxs = None
        # For condition D, build a context list
        if use_context:
            # Put the correct passage first, followed by random distractor passages
            ctxs = [r["chunk"]] + [kb_texts[rng.randrange(len(kb_texts))] for _ in range(n_distractors)]
            # Shuffle so the correct passage is not always in the same position
            rng.shuffle(ctxs)
        # Build the full chat-formatted prompt text for this question (and context, if any)
        prompt = tokenizer.apply_chat_template(
            # A single user message containing the instructions, context and question
            [{"role": "user", "content": build_prompt(r["question"], ctxs)}],
            # Add the marker that tells the model its answer starts here; return text, not tokens
            add_generation_prompt=True, tokenize=False)
        # Convert the prompt text into token IDs (no extra special tokens, the template already has them)
        p = tokenizer(prompt, add_special_tokens=False)["input_ids"]
        # Convert the reference answer, plus the end-of-turn marker, into token IDs
        a = tokenizer(r["answer"] + "<end_of_turn>\n", add_special_tokens=False)["input_ids"]
        # If prompt + answer is too long, the answer would be cut off, so skip this example
        if len(p) + len(a) > max_len:
            # Count the skipped example
            dropped += 1
            # Move on to the next row
            continue
        # Keep the example: labels of -100 on the prompt mean the loss is computed only on the answer
        exs.append({"input_ids": p + a, "labels": [-100] * len(p) + a})
    # Report how many examples were kept and how many were skipped
    print(f"kept {len(exs)} examples, dropped {dropped} that exceeded {max_len} tokens")
    # Return the list of training examples
    return exs


# Combine several examples into one padded batch for the trainer
def collate(batch):
    # Find the longest example in the batch, and get the padding token ID
    m = max(len(b["input_ids"]) for b in batch); pad = tokenizer.pad_token_id
    # Return the three tensors the model needs
    return {
        # Token IDs, padded on the right up to the longest length
        "input_ids": torch.tensor([b["input_ids"] + [pad] * (m - len(b["input_ids"])) for b in batch]),
        # Attention mask: 1 for real tokens, 0 for padding
        "attention_mask": torch.tensor([[1] * len(b["input_ids"]) + [0] * (m - len(b["input_ids"])) for b in batch]),
        # Labels padded with -100 so padding is ignored in the loss
        "labels": torch.tensor([b["labels"] + [-100] * (m - len(b["labels"])) for b in batch]),
    }


# Train one LoRA adapter and save it to out_dir; returns the training time in seconds
def train_lora(use_context, out_dir, n_train=None, epochs=2, rank=16):
    # Free any leftover memory before starting
    gc.collect()
    # Also release cached GPU memory if we are on an NVIDIA GPU
    if DEVICE == "cuda": torch.cuda.empty_cache()
    # Take the first n_train training rows, or all of them if n_train is None
    rows = [train[i] for i in range(len(train) if n_train is None else n_train)]
    # Convert those rows into tokenised examples
    exs = make_examples(rows, use_context)
    # Load the base Gemma model in 32-bit precision (stable on a T4) with eager attention (recommended for Gemma 3)
    model = AutoModelForCausalLM.from_pretrained(
        MODEL, torch_dtype=torch.float32, attn_implementation="eager").to(DEVICE)
    # Turn off the generation cache, which is not needed (and conflicts) during training
    model.config.use_cache = False
    # Gradient checkpointing: recompute activations instead of storing them, saving a lot of GPU memory
    model.gradient_checkpointing_enable(gradient_checkpointing_kwargs={"use_reentrant": False})
    # Needed so gradients flow properly when the base model is frozen and checkpointing is on
    model.enable_input_require_grads()
    # Describe the LoRA adapter: rank, scaling, dropout, task, and which layers to adapt
    cfg = LoraConfig(r=rank, lora_alpha=2 * rank, lora_dropout=0.05, task_type="CAUSAL_LM",
                     target_modules=["q_proj", "k_proj", "v_proj", "o_proj", "gate_proj", "up_proj", "down_proj"])
    # Wrap the model so only the small LoRA layers are trainable
    model = get_peft_model(model, cfg)
    # Print how many parameters are being trained
    model.print_trainable_parameters()

    # One example per step, accumulated over 16 steps = effective batch size of 16
    batch, accum = 1, 16
    # Total number of optimiser updates over all epochs
    total_steps = math.ceil(len(exs) / (batch * accum)) * epochs
    # Use 5% of the steps for learning-rate warmup (at least 1)
    warmup_steps = max(1, int(0.05 * total_steps))

    # Collect all training settings
    args = TrainingArguments(
        # Folder for trainer files
        output_dir=out_dir,
        # Examples processed at once on the GPU
        per_device_train_batch_size=batch,
        # Steps to accumulate before each weight update
        gradient_accumulation_steps=accum,
        # Number of passes over the training data
        num_train_epochs=epochs,
        # Learning rate, with a cosine decay schedule
        learning_rate=2e-4, lr_scheduler_type="cosine",
        # Warmup steps, and how often to print the loss
        warmup_steps=warmup_steps, logging_steps=10,
        # Do not save checkpoints during training (we save the adapter at the end)
        save_strategy="no",
        # Disable external logging services; keep our custom columns
        report_to="none", remove_unused_columns=False)
    # Record the start time
    t = time.time()
    # Create the trainer with the model, settings, data and batching function
    trainer = Trainer(model=model, args=args, train_dataset=exs, data_collator=collate)
    # Run the training loop
    trainer.train()
    # Compute how long training took
    secs = time.time() - t

    # Save the trained LoRA adapter to disk
    model.save_pretrained(out_dir)
    # Confirm the save by listing the folder contents
    print("saved to:", os.path.abspath(out_dir), os.listdir(out_dir))

    # Delete the trainer and model to release memory
    del trainer, model
    # Run garbage collection
    gc.collect()
    # Release cached GPU memory
    if DEVICE == "cuda": torch.cuda.empty_cache()
    # Return the training time
    return secs


# Number of training pairs to use: 200 for a quick test, or None for all (about 1,700)
N_TRAIN = 200
# Dictionary to store training times for the results table
train_times = {}

# Loop over the two adapters: C = no context, D = with context
for name, use_ctx in [("C", False), ("D", True)]:
    # Folder name for this adapter, e.g. adapter_C
    out_dir = f"adapter_{name}"
    # If the adapter file already exists, do not train again
    if os.path.exists(os.path.join(out_dir, "adapter_model.safetensors")):
        # Tell the user it is being skipped
        print(f"{out_dir} already saved, skipping")
        # No training time to record for a skipped adapter
        train_times[name] = None
        # Go to the next adapter
        continue
    # Otherwise train it and record the time
    train_times[name] = train_lora(use_context=use_ctx, out_dir=out_dir, n_train=N_TRAIN)

# Show the training times
print(train_times)

Using device: cuda
kept 200 examples, dropped 0 that exceeded 1024 tokens


Loading weights:   0%|          | 0/340 [00:00<?, ?it/s]

trainable params: 13,045,760 || all params: 1,012,931,712 || trainable%: 1.2879


Step,Training Loss
10,2.263569
20,1.824060


saved to: /content/adapter_C ['README.md', 'adapter_model.safetensors', 'adapter_config.json']
kept 191 examples, dropped 9 that exceeded 1024 tokens


Loading weights:   0%|          | 0/340 [00:00<?, ?it/s]

trainable params: 13,045,760 || all params: 1,012,931,712 || trainable%: 1.2879


Step,Training Loss
10,0.570078
20,0.307078


saved to: /content/adapter_D ['README.md', 'adapter_model.safetensors', 'adapter_config.json']
{'C': 196.20651364326477, 'D': 612.4433557987213}


## 5. Evaluation
Metrics per condition: ROUGE-L and embedding similarity vs the reference answer, latency (retrieval + generation), new tokens, peak GPU memory, training time.

**Extensions for higher marks** (add to your report): LLM-as-judge for correctness and faithfulness (Zheng et al., 2023), an "unanswerable" test set (questions whose gold chunk is removed from the KB) to measure whether the model correctly declines, and k / chunking ablations.

In [7]:
# rouge_scorer measures word overlap between two texts
from rouge_score import rouge_scorer
# Create a ROUGE-L scorer (with stemming so "sentence" and "sentenced" match)
rs = rouge_scorer.RougeScorer(["rougeL"], use_stemmer=True)

# Run one condition on a set of questions and return a table of results
def run_condition(name, model, rows, retrieve=None, k=3):
    # List that collects one record per question
    recs = []
    # Reset the GPU memory counter so we measure the peak for this condition only
    torch.cuda.reset_peak_memory_stats()
    # Go through each question
    for r in rows:
        # By default no passages and zero search time
        ctxs, rt = None, 0.0
        # If a search function was given (RAG conditions)
        if retrieve:
            # Time the search and keep the top k passage IDs
            t = time.time(); ids = retrieve(r["question"])[:k]; rt = time.time() - t
            # Look up the text of those passages
            ctxs = [kb_texts[i] for i in ids]
        # Generate the answer; get the text, the generation time and the token count
        pred, dt, nt = generate(model, build_prompt(r["question"], ctxs))
        # Store everything about this question
        recs.append(dict(condition=name, question=r["question"], reference=r["answer"], prediction=pred,
                         # Total time = search time + generation time
                         latency=rt + dt, new_tokens=nt,
                         # ROUGE-L F-score between the reference answer and the model's answer
                         rougeL=rs.score(r["answer"], pred)["rougeL"].fmeasure))
    # Turn the list of records into a table
    df = pd.DataFrame(recs)
    # Embed all the model answers
    pe = emb_model.encode(df.prediction.tolist(), normalize_embeddings=True)
    # Embed all the reference answers
    re_ = emb_model.encode(df.reference.tolist(), normalize_embeddings=True)
    # Cosine similarity of each pair (vectors are normalised, so a dot product is enough)
    df["emb_sim"] = (pe * re_).sum(1)
    # Peak GPU memory used during this condition, in gigabytes
    df["peak_gb"] = torch.cuda.max_memory_allocated() / 1e9
    # Return the table for this condition
    return df

# How many test questions to evaluate (50 here to save time)
N_EVAL = 50     # increase (e.g. 100-425) for final results
# Take the first N_EVAL test questions
rows = test_rows[:N_EVAL]
# List that collects the results tables of all conditions
results = []

# Load the plain base model (used for conditions A and B)
m = load_model()
# Condition A: base model, no passages
results.append(run_condition("A_base", m, rows))
# Condition B: base model with RAG (hybrid search + rerank, top 3 passages)
results.append(run_condition("B_rag", m, rows, retrieve=hybrid_rerank))
# Free the base model from the GPU
del m; torch.cuda.empty_cache()

# Load the base model with adapter C (fine-tuned, no context)
m = load_model("adapter_C")
# Condition C: fine-tuned model, no passages
results.append(run_condition("C_finetuned", m, rows))
# Free the GPU again
del m; torch.cuda.empty_cache()

# Load the base model with adapter D (fine-tuned with context)
m = load_model("adapter_D")
# Condition D: fine-tuned model with RAG
results.append(run_condition("D_finetuned_rag", m, rows, retrieve=hybrid_rerank))
# Free the GPU again
del m; torch.cuda.empty_cache()

# Stack the four result tables into one big table
all_results = pd.concat(results)
# Average the scores for each condition
summary = all_results.groupby("condition")[["rougeL", "emb_sim", "latency", "new_tokens", "peak_gb"]].mean()
# Add training times (zero for A and B because they were not trained)
summary["train_secs"] = [0, 0, train_times["C"], train_times["D"]]
# Show the summary table
summary

Loading weights:   0%|          | 0/340 [00:00<?, ?it/s]

Loading weights:   0%|          | 0/340 [00:00<?, ?it/s]

Loading weights:   0%|          | 0/340 [00:00<?, ?it/s]

,rougeL,emb_sim,latency,new_tokens,peak_gb,train_secs
condition,,,,,,
A_base,0.276945,0.872895,9.143684,195.18,4.407400,0.000000
B_rag,0.436712,0.844490,3.668414,65.48,4.557143,0.000000
C_finetuned,0.367942,0.887729,12.022961,155.16,4.459583,196.206514
D_finetuned_rag,0.577661,0.933580,9.455445,116.98,4.609326,612.443356


In [8]:
# Save everything for the report / GitHub repo
# Save every question's answer and scores
all_results.to_csv("generation_results.csv", index=False)
# Save the averaged summary table
summary.to_csv("summary.csv")
# Save the search results table
retr_results.to_csv("retrieval_results.csv")

# Look at failures
# For each condition, pick the 2 answers with the lowest ROUGE-L
worst = all_results.sort_values("rougeL").groupby("condition").head(2)
# Go through those rows one by one
for _, r in worst.iterrows():
    # Print the condition, the question, the reference answer and the model's answer
    print(f"[{r.condition}] Q: {r.question}\nREF: {r.reference}\nPRED: {r.prediction}\n")

[B_rag] Q: What was the decision of the court in the case JMS v R [2010] NSWCCA 229 in New South Wales?
REF: In the case JMS v R [2010] NSWCCA 229, the court in New South Wales granted leave to appeal, allowed the appeal, and quashed the sentences below. Instead, the court ordered that the applicant be sentenced to imprisonment on count 1 for three and a half years commencing on 14 October 2009, expiring on 13 April 2013, with a non-parole period of eighteen months commencing 14 October 2009 and expiring on 13 April 2011; and on count 2, to imprisonment for three and a half years commencing 14 April 2010 and expiring on 13 October 2013, with a non-parole period of eighteen months commencing 14 April 2010 and expiring on 13 October 2011.
PRED: The context does not contain the answer to the question.

[B_rag] Q: In the case of People for the Plains Incorporated v Santos NSW (Eastern) [2015] NSWSC 1792, what was the court's decision regarding the jurisdiction of the Land and Environment C

In [9]:
# gc lets us force Python to release unused memory
import gc, os, re, time, shutil
# numpy and pandas for numbers and tables, torch for the GPU
import numpy as np, pandas as pd, torch

# A small function that frees GPU memory between models
def free_gpu():
    # Ask Python to delete unused objects
    gc.collect()
    # If an NVIDIA GPU exists, also release its cached memory
    if torch.cuda.is_available(): torch.cuda.empty_cache()

# Part 2: improvements


What this adds, ordered cheapest first:

| Step | What | Rough time on a T4 |
|---|---|---|
| 0 | Restore/check the 200-example adapters | seconds (about 15 min only if adapters must be retrained) |
| 1 | Choose the search method on **training** questions (not the test set), including weighted fusion and the BGE query prefix | about 1 minute |
| 2 | Re-run conditions B and D with the chosen search (fixes the reranker flaw) and split scores by whether the right passage was found | about 10-12 minutes |
| 3 | Citation and groundedness checks on the answers | seconds |
| 4 | Unanswerable-question test (correct passage removed) | about 6-8 minutes |
| 5 | Confidence intervals and saving results | seconds |

T

In [10]:
# os and shutil handle folders, gc frees memory, time measures speed, re does text patterns
import os, shutil, gc, time, re
# numpy and pandas for numbers and tables, torch for the GPU
import numpy as np, pandas as pd, torch

# Helper that frees GPU memory between models
def free_gpu():
    # Ask Python to delete unused objects
    gc.collect()
    # Release cached GPU memory
    torch.cuda.empty_cache()

# If the optional scale-up cell renamed your adapters, restore the 200-example ones used in the report
for n in ("adapter_C", "adapter_D"):
    # Check whether a renamed "_smoke" copy exists
    if os.path.exists(n + "_smoke"):
        # Remove any half-finished replacement folder
        if os.path.exists(n): shutil.rmtree(n)
        # Move the original small-run adapter back to its normal name
        shutil.move(n + "_smoke", n)

# True if a finished adapter file exists in this folder
def has_adapter(n): return os.path.exists(os.path.join(n, "adapter_model.safetensors"))

# Make sure the dictionary of training times exists
train_times = globals().get("train_times", {})
# Check both adapters
for n, use_ctx in (("adapter_C", False), ("adapter_D", True)):
    # If an adapter is missing (for example after a Colab reset), train it again on 200 pairs
    if not has_adapter(n):
        # Tell the user what is happening
        print("missing", n, "- retraining on 200 pairs (a few minutes)")
        # Train and record the time
        train_times[n[-1]] = train_lora(use_context=use_ctx, out_dir=n, n_train=200)
# Show the adapter status
print({n: has_adapter(n) for n in ("adapter_C", "adapter_D")})

{'adapter_C': True, 'adapter_D': True}


## 1. Choose the search method on training questions
In the main notebook I picked the reranked search before measuring it. Here the choice is made on **300 training questions** and only then reported on the test set, so the test set does not influence the choice. New candidates: a query instruction prefix for BGE (recommended for its search queries), and weighted fusion that gives BM25 more weight, since keyword search did best on this legal text.

In [11]:
# 300 training questions used only for choosing the search method
dev_rows = [train[i] for i in range(300)]
# Instruction text that the BGE model suggests putting in front of search queries
PREFIX = "Represent this sentence for searching relevant passages: "

# Dense search where the question has the instruction prefix
def dense_prefixed(q, k=100):
    # Turn the prefixed question into a vector
    qe = emb_model.encode([PREFIX + q], normalize_embeddings=True)[0]
    # Return the IDs of the k most similar passages
    return np.argsort(-(kb_emb @ qe))[:k].tolist()

# Run each basic search once per question and keep the top 100 results of each
def precompute(rows):
    # One record per question: the correct passage ID plus three ranked lists
    return [{"gold": r["kb_id"], "q": r["question"],
             # Keyword ranking
             "bm25": bm25_rank(r["question"], 100),
             # Embedding ranking
             "dense": dense_rank(r["question"], 100),
             # Embedding ranking with the query prefix
             "dense_pref": dense_prefixed(r["question"], 100)} for r in rows]

# Weighted Reciprocal Rank Fusion: each list adds weight/(c + position) to a passage's score
def fuse(lists_weights, k=10, c=60):
    # Dictionary of passage ID -> combined score
    scores = {}
    # Go through each ranked list with its weight
    for ranking, w in lists_weights:
        # pos = position in the list (0 is best)
        for pos, i in enumerate(ranking):
            # Add this list's contribution to the passage's score
            scores[i] = scores.get(i, 0) + w / (c + pos + 1)
    # Return the top k passage IDs by combined score
    return sorted(scores, key=scores.get, reverse=True)[:k]

# The search methods to compare; each takes the precomputed lists and returns 10 passage IDs
CONFIGS = {
    # Keyword search only
    "bm25": lambda p: p["bm25"][:10],
    # Embedding search only
    "dense": lambda p: p["dense"][:10],
    # Embedding search with the query prefix
    "dense+prefix": lambda p: p["dense_pref"][:10],
    # Equal-weight fusion (this was the original hybrid)
    "rrf 1:1": lambda p: fuse([(p["bm25"], 1), (p["dense"], 1)]),
    # BM25 counts twice as much as dense
    "rrf 2:1": lambda p: fuse([(p["bm25"], 2), (p["dense"], 1)]),
    # BM25 counts three times as much as dense
    "rrf 3:1": lambda p: fuse([(p["bm25"], 3), (p["dense"], 1)]),
    # Weighted fusion using the prefixed dense search
    "rrf 2:1 +prefix": lambda p: fuse([(p["bm25"], 2), (p["dense_pref"], 1)]),
}

# Score one search method: recall@k and mean reciprocal rank
def score_cfg(pre, fn, ks=(1, 3, 5)):
    # Position of the correct passage for each question (None if not found)
    ranks = []
    # Go through each question
    for p in pre:
        # Get this method's top 10 passage IDs
        r = fn(p)
        # Record the 1-based position of the correct passage, or None
        ranks.append(r.index(p["gold"]) + 1 if p["gold"] in r else None)
    # Number of questions
    n = len(pre)
    # Recall@k = share of questions with the correct passage in the top k
    out = {f"recall@{k}": sum(1 for x in ranks if x and x <= k) / n for k in ks}
    # Mean reciprocal rank
    out["MRR"] = sum(1 / x for x in ranks if x) / n
    # Return the scores
    return out

# Run the searches on the development questions
pre_dev = precompute(dev_rows)
# Score every method and sort by recall@3
dev_table = pd.DataFrame({name: score_cfg(pre_dev, fn) for name, fn in CONFIGS.items()}).T
# Show the development results (this is where the choice is made)
dev_table.sort_values(["recall@3", "MRR"], ascending=False)

,recall@1,recall@3,recall@5,MRR
bm25,0.753333,0.830000,0.856667,0.797022
rrf 1:1,0.770000,0.826667,0.843333,0.802101
rrf 3:1,0.783333,0.823333,0.836667,0.811537
rrf 2:1 +prefix,0.783333,0.816667,0.843333,0.809710
rrf 2:1,0.780000,0.813333,0.840000,0.808151
dense,0.706667,0.786667,0.806667,0.749921
dense+prefix,0.713333,0.780000,0.790000,0.749882


In [12]:
# Pick the method with the best recall@3 on the development questions (ties broken by MRR)
best_name = dev_table.sort_values(["recall@3", "MRR"], ascending=False).index[0]
# Run the searches on all test questions
pre_test = precompute(test_rows)
# Score every method on the test set, for the report (the choice was NOT made here)
test_table = pd.DataFrame({name: score_cfg(pre_test, fn) for name, fn in CONFIGS.items()}).T
# Say which method was chosen
print("chosen on development questions:", best_name)
# Show the test results
test_table.sort_values(["recall@3", "MRR"], ascending=False)

chosen on development questions: bm25


,recall@1,recall@3,recall@5,MRR
rrf 2:1 +prefix,0.807059,0.887059,0.903529,0.850984
rrf 2:1,0.802353,0.882353,0.903529,0.845833
rrf 3:1,0.804706,0.880000,0.903529,0.850057
rrf 1:1,0.807059,0.870588,0.896471,0.842254
bm25,0.776471,0.870588,0.894118,0.830645
dense+prefix,0.738824,0.809412,0.835294,0.781200
dense,0.731765,0.809412,0.835294,0.776133


In [13]:
# The final search function used for conditions B2 and D2: returns the top k passage IDs
def best_retrieve(q, k=3):
    # Run the three basic searches for this question
    p = {"bm25": bm25_rank(q, 100), "dense": dense_rank(q, 100), "dense_pref": dense_prefixed(q, 100)}
    # Apply the chosen method and keep the top k
    return CONFIGS[best_name](p)[:k]

## 2. Re-run conditions B and D with the chosen search
Same model, prompts and 50 test questions as before. The new function also records the correct passage ID and which passages were given, so we can split the scores by whether the right passage was found.

In [14]:
# ROUGE scorer from the main notebook is reused as "rs"; this function runs one condition and records extra details
def run_condition2(name, model, rows, retrieve=None, k=3, exclude_gold=False):
    # List that collects one record per question
    recs = []
    # Go through each question
    for r in rows:
        # Start with no passages and no search time
        ids, rt = [], 0.0
        # If a search function was given (RAG conditions)
        if retrieve:
            # Start timing the search
            t = time.time()
            # Ask for one extra result when we plan to remove the correct passage
            ids = retrieve(r["question"], k + 1 if exclude_gold else k)
            # For the unanswerable test, drop the correct passage from the results
            if exclude_gold: ids = [i for i in ids if i != r["kb_id"]]
            # Keep only k passages
            ids = ids[:k]
            # Search time
            rt = time.time() - t
        # Look up the passage texts (or None if no search)
        ctxs = [kb_texts[i] for i in ids] if ids else None
        # Generate the answer
        pred, dt, nt = generate(model, build_prompt(r["question"], ctxs))
        # Store everything about this question
        recs.append(dict(condition=name, question=r["question"], reference=r["answer"], prediction=pred,
                         # ID of the correct passage and the IDs actually given to the model
                         gold=r["kb_id"], ctx_ids=ids,
                         # True if the correct passage was among those given
                         gold_in_ctx=(r["kb_id"] in ids),
                         # Search time plus generation time, and number of new tokens
                         latency=rt + dt, new_tokens=nt,
                         # ROUGE-L against the reference answer
                         rougeL=rs.score(r["answer"], pred)["rougeL"].fmeasure))
    # Turn the records into a table
    df = pd.DataFrame(recs)
    # Embed the model answers
    pe = emb_model.encode(df.prediction.tolist(), normalize_embeddings=True)
    # Embed the reference answers
    re_ = emb_model.encode(df.reference.tolist(), normalize_embeddings=True)
    # Cosine similarity of each pair
    df["emb_sim"] = (pe * re_).sum(1)
    # Return the table
    return df

# The same 50 test questions as the main evaluation
rows = test_rows[:50]
# Load the base model
m = load_model()
# Condition B2: base model + RAG with the chosen search
B2 = run_condition2("B2_rag_chosen_search", m, rows, retrieve=best_retrieve)
# Free the GPU
del m; free_gpu()
# Load the base model with adapter D
m = load_model("adapter_D")
# Condition D2: fine-tuned model + RAG with the chosen search
D2 = run_condition2("D2_finetuned_rag_chosen_search", m, rows, retrieve=best_retrieve)
# Free the GPU
del m; free_gpu()
# Put both results in one table
new_results = pd.concat([B2, D2])
# Average scores per condition, plus time per generated token
summary2 = new_results.groupby("condition")[["rougeL", "emb_sim", "latency", "new_tokens", "gold_in_ctx"]].mean()
# Milliseconds per generated token (a fairer speed measure than time per question)
summary2["ms_per_token"] = summary2["latency"] / summary2["new_tokens"] * 1000
# Show the summary
summary2

Loading weights:   0%|          | 0/340 [00:00<?, ?it/s]

Loading weights:   0%|          | 0/340 [00:00<?, ?it/s]

,rougeL,emb_sim,latency,new_tokens,gold_in_ctx,ms_per_token
condition,,,,,,
B2_rag_chosen_search,0.427571,0.846747,3.594903,69.14,0.88,51.994554
D2_finetuned_rag_chosen_search,0.537325,0.928326,9.519248,123.80,0.88,76.892149


## 3. Does the answer depend on finding the right passage? Citations and groundedness


In [15]:
# Average scores split by condition and by whether the correct passage was given to the model
split_table = new_results.groupby(["condition", "gold_in_ctx"])[["rougeL", "emb_sim"]].agg(["mean", "count"])
# Show it
print(split_table)

# True if the answer cites a passage number that points to the correct passage
def cites_gold(row):
    # Find numbers written like [1], [2] in the answer
    nums = [int(x) for x in re.findall(r"\[(\d+)\]", row.prediction)]
    # Check if any cited number maps to the correct passage ID
    return any(1 <= n <= len(row.ctx_ids) and row.ctx_ids[n - 1] == row.gold for n in nums)

# Share of the answer's longer words that also appear in the given passages
def grounded_frac(row):
    # All the passage text joined together, in lower case
    ctx = " ".join(kb_texts[i] for i in row.ctx_ids).lower()
    # Words of four or more letters in the answer
    words = re.findall(r"[a-z]{4,}", row.prediction.lower())
    # Fraction of those words found in the passages (NaN if there are no words)
    return float(np.mean([w in ctx for w in words])) if words else np.nan

# Does the answer contain any citation like [1]?
new_results["has_citation"] = new_results.prediction.str.contains(r"\[\d+\]")
# Does it cite the correct passage?
new_results["cites_gold"] = new_results.apply(cites_gold, axis=1)
# Word-overlap groundedness proxy
new_results["grounded"] = new_results.apply(grounded_frac, axis=1)
# Average these per condition
new_results.groupby("condition")[["has_citation", "cites_gold", "grounded"]].mean()

                                              rougeL         emb_sim      
                                                mean count      mean count
condition                      gold_in_ctx                                
B2_rag_chosen_search           False        0.255285     6  0.769425     6
                               True         0.451065    44  0.857291    44
D2_finetuned_rag_chosen_search False        0.406977     6  0.877123     6
                               True         0.555100    44  0.935309    44


,has_citation,cites_gold,grounded
condition,,,
B2_rag_chosen_search,0.16,0.0,0.813878
D2_finetuned_rag_chosen_search,0.74,0.0,0.874604


## 4. Unanswerable-question test
For each question the correct passage is **removed** from the search results, so the answer is not in the context. A careful system should decline. We compare how often each condition declines here (want high) with how often it declines on answerable questions (want low). Decline detection is a keyword pattern, so **read some outputs by hand**.

In [16]:
# Phrases that suggest the model is declining to answer
DECLINE = re.compile(r"cannot answer|can't answer|can not answer|not contain|does not (provide|mention|include|specify|say)|"
                     r"do not (provide|contain)|unable to|no information|not enough information|insufficient", re.I)
# True if the text contains one of those phrases
def declined(text): return bool(DECLINE.search(text))

# Use 30 questions to keep the run short
rows_u = test_rows[:30]
# Load the base model
m = load_model()
# Base model + RAG, with the correct passage removed from the results
U_B = run_condition2("B2_unanswerable", m, rows_u, retrieve=best_retrieve, exclude_gold=True)
# Free the GPU
del m; free_gpu()
# Load the base model with adapter D
m = load_model("adapter_D")
# Fine-tuned model + RAG, with the correct passage removed
U_D = run_condition2("D2_unanswerable", m, rows_u, retrieve=best_retrieve, exclude_gold=True)
# Free the GPU
del m; free_gpu()
# Combine the two tables and mark which answers declined
unans = pd.concat([U_B, U_D]); unans["declined"] = unans.prediction.map(declined)
# Decline rates: when no answer exists (want high) and when the answer was available (want low)
decline_rates = pd.DataFrame({
    # Share that declined when the correct passage was removed
    "decline_when_no_answer (want high)": [U_B.prediction.map(declined).mean(), U_D.prediction.map(declined).mean()],
    # Share that declined on the normal answerable run
    "decline_when_answerable (want low)": [B2.prediction.map(declined).mean(), D2.prediction.map(declined).mean()],
}, index=["B2 (base + RAG)", "D2 (fine-tuned + RAG)"])
# Show the rates
print(decline_rates)
# Show 8 random outputs so you can check the keyword detection by hand
unans.sample(8, random_state=0)[["condition", "question", "prediction", "declined"]]

Loading weights:   0%|          | 0/340 [00:00<?, ?it/s]

Loading weights:   0%|          | 0/340 [00:00<?, ?it/s]

                       decline_when_no_answer (want high)  \
B2 (base + RAG)                                  0.433333   
D2 (fine-tuned + RAG)                            0.000000   

                       decline_when_answerable (want low)  
B2 (base + RAG)                                      0.10  
D2 (fine-tuned + RAG)                                0.02  


,condition,question,prediction,declined
26,B2_unanswerable,In the case of The Owners – Strata Plan No. 92...,The context does not contain information about...,True
5,D2_unanswerable,What is the purpose of the general purpose fin...,In the Federal Financial Relations (General pu...,False
29,D2_unanswerable,What was the sentence given to Stephen Edward ...,"In the case REGINA v SAARI [2001] NSWSC 892, S...",False
28,B2_unanswerable,In the case of Dutta v Telstra Corporation Lim...,The provided text does not contain any informa...,True
11,B2_unanswerable,What are the modifications required for the el...,"According to the provided text, the modificati...",False
2,B2_unanswerable,In the case of R v James Nicholas Kontaxis [20...,"According to the provided text, the factors co...",False
4,D2_unanswerable,Who were the parties involved in the case Prop...,In the case of Propertylink Finance Pty Ltd ac...,False
28,D2_unanswerable,In the case of Dutta v Telstra Corporation Lim...,In the case of Dutta v Telstra Corporation Lim...,False


## 5. Confidence intervals and saving
Paired bootstrap on ROUGE-L: resample the questions 5,000 times to get a 95% interval for the average difference between two conditions. If the interval includes 0, do not claim one is better.

In [17]:
# One row per question, one column per condition, holding ROUGE-L (new runs)
piv = new_results.pivot_table(index="question", columns="condition", values="rougeL")
# If the main notebook's results are still in memory, add the original conditions too
if "all_results" in globals():
    # Same layout for the original four conditions
    piv_old = all_results.pivot_table(index="question", columns="condition", values="rougeL")
    # Join the two tables on the question text
    piv = piv.join(piv_old, how="left")

# Bootstrap the mean difference a - b
def boot(a, b, n=5000, seed=0):
    # Per-question differences, ignoring questions where either score is missing
    d = (piv[a] - piv[b]).dropna().values
    # Random generator with a fixed seed so results repeat
    rng = np.random.default_rng(seed)
    # Resample the differences with replacement and average each time
    means = [rng.choice(d, len(d)).mean() for _ in range(n)]
    # Middle 95% range of those averages
    lo, hi = np.percentile(means, [2.5, 97.5])
    # Return the result as one row
    return {"A": a, "B": b, "mean_diff": d.mean(), "ci95_low": lo, "ci95_high": hi, "n": len(d)}

# Comparisons to run (only those whose columns exist)
wanted = [("D2_finetuned_rag_chosen_search", "B2_rag_chosen_search"),
          ("B2_rag_chosen_search", "A_base"), ("D2_finetuned_rag_chosen_search", "C_finetuned"),
          ("D2_finetuned_rag_chosen_search", "A_base")]
# Keep the comparisons that are possible
pairs = [(a, b) for a, b in wanted if a in piv.columns and b in piv.columns]
# Compute and show them
ci_table = pd.DataFrame([boot(a, b) for a, b in pairs])
ci_table

,A,B,mean_diff,ci95_low,ci95_high,n
0,D2_finetuned_rag_chosen_search,B2_rag_chosen_search,0.109754,0.060141,0.158272,50
1,B2_rag_chosen_search,A_base,0.150626,0.099776,0.199172,50
2,D2_finetuned_rag_chosen_search,C_finetuned,0.169384,0.120008,0.217948,50
3,D2_finetuned_rag_chosen_search,A_base,0.260380,0.211152,0.310372,50


In [18]:
# Folder for the new results
os.makedirs("results_part2", exist_ok=True)
# Save each table as a CSV file
for name, obj in {"search_dev": dev_table, "search_test": test_table, "answers_summary": summary2,
                  "answers_split_by_gold": split_table, "answers_all": new_results,
                  "unanswerable": unans, "decline_rates": decline_rates, "confidence_intervals": ci_table}.items():
    # Write results_part2/<name>.csv
    obj.to_csv(f"results_part2/{name}.csv")
# Zip the folder so it is one easy download
shutil.make_archive("results_part2", "zip", "results_part2")
# List the files created
print(os.listdir("results_part2"))

['confidence_intervals.csv', 'search_test.csv', 'answers_split_by_gold.csv', 'decline_rates.csv', 'unanswerable.csv', 'answers_all.csv', 'answers_summary.csv', 'search_dev.csv']
